# Compliance & Audit Workbench
See which obligations apply for any entity selection, review rule results, verify ledger integrity, and build an evidence extract.

In [1]:
import json, pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
from meridian.pipelines.target_date import run
from meridian.policy import load_policy, load_universe
from meridian import entities
from meridian.ledger import verify, Ledger

## 1. What changes with each operating entity?

In [2]:
rows = {}
for k in entities.load_packs():
    e = entities.resolve([k])
    rows[k] = {'retention_yrs': e.retention_years, 'obligations': len(e.obligations), 'required_roles': ', '.join(e.required_roles)}
pd.DataFrame(rows).T

,retention_yrs,obligations,required_roles
sec_ria,5,7,"chief_compliance_officer, portfolio_manager"
broker_dealer,6,6,"chief_compliance_officer, registered_principal"
registered_fund,6,7,"chief_compliance_officer, derivatives_risk_man..."
private_fund,5,2,chief_compliance_officer
erisa_plan,6,4,plan_fiduciary_committee
bank_cit,6,2,trust_committee
pension_endowment_ocio,7,1,investment_committee
cftc_derivatives,5,1,


In [3]:
current = entities.resolve()
print('Selected:', current.selected)
pd.DataFrame(current.obligations)[['id','cite','summary','regime']]

Selected: ['sec_ria', 'registered_fund', 'erisa_plan', 'cftc_derivatives']


,id,cite,summary,regime
0,IA-FID,Advisers Act §206; SEC 2019 fiduciary interpre...,Duty of care and loyalty; best interest; full ...,sec_ria
1,IA-CP,Rule 206(4)-7,"Written compliance policies and procedures, an...",sec_ria
2,IA-REC,Rule 204-2,"Books and records, including records supportin...",sec_ria
3,IA-MKT,Rule 206(4)-1,Marketing rule; hypothetical performance (back...,sec_ria
4,IA-ADV,Form ADV Part 2A,Disclose methods of analysis incl. use of AI/a...,sec_ria
5,IA-COE,Rule 204A-1,Code of ethics; personal trading,sec_ria
6,SP-PRIV,Regulation S-P (2024 amendments),"Safeguards, incident response, customer notifi...",sec_ria
7,IC-CP,Rule 38a-1,Fund compliance program; board-approved; CCO r...,registered_fund
8,IC-DIV,ICA §5(b)(1),Diversified fund 75-5-10 test (look-through no...,registered_fund
9,IC-FOF,ICA §12(d)(1); Rule 12d1-4,Fund-of-funds acquisition limits / conditions,registered_fund


## 2. Rule results

In [4]:
res = run(as_of='2026-10-31')
matrix = pd.DataFrame({v: {r['rule_id']: r['status'] for r in d['compliance']} for v, d in res.vintages.items()})
matrix

,2030,2035,2040,2045,2050,2055,2060,2065,2070
POL-SUM,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
POL-LONG-ONLY,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
POL-BAND-ASSET,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
POL-BAND-EQUITY,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
POL-BAND-GROWTH,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
CONC-BELOW-IG,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
CONC-NONUS-EQ,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
CONC-CASH,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
CONC-SINGLE-FUND,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS
TE-NEUTRAL,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS,PASS


In [5]:
print((res.out_dir/'compliance_report.md').read_text()[:2500])

# Compliance Review — run_tds_20261031_c5fcf3

**Prepared by:** Agentic Compliance Officer (ACO) — AI agent. **Status: DRAFT.** Not an attestation by a Chief Compliance Officer.

**Overall status:** CONDITIONAL
**Ledger integrity:** OK — verified 116 records; head ff5d1f9b1d33

## Entity regimes in force

- `sec_ria`
- `registered_fund`
- `erisa_plan`
- `cftc_derivatives`

Record retention applied: **6 years** (strictest selected regime).

## Obligations mapped

| ID | Citation | Requirement | Regime |
|---|---|---|---|
| IA-FID | Advisers Act §206; SEC 2019 fiduciary interpretation | Duty of care and loyalty; best interest; full and fair disclosure of conflicts | sec_ria |
| IA-CP | Rule 206(4)-7 | Written compliance policies and procedures, annual review, designated CCO (a natural person) | sec_ria |
| IA-REC | Rule 204-2 | Books and records, including records supporting recommendations and performance | sec_ria |
| IA-MKT | Rule 206(4)-1 | Marketing rule; hypothetical performance (b

## 3. Ledger integrity — and tamper detection
Every record is hash-chained. Altering any record breaks verification.

In [6]:
print(verify(res.out_dir/'ledger.jsonl'))
import shutil, tempfile, pathlib
tmp = pathlib.Path(tempfile.mkdtemp())/'ledger.jsonl'; shutil.copy(res.out_dir/'ledger.jsonl', tmp)
lines = tmp.read_text().splitlines(); rec = json.loads(lines[10]); rec['summary'] = 'edited after the fact'; lines[10] = json.dumps(rec)
tmp.write_text('\n'.join(lines)+'\n'); print(verify(tmp))

(True, 'verified 118 records; head 72956250f9b3')
(False, 'record 10 (dr_62187b099a83418cb7f1) content altered')


## 4. Evidence extract for one vintage
Everything that led to the 2050 proposal, in order.

In [7]:
recs = Ledger(res.out_dir/'ledger.jsonl').records()
ev = pd.DataFrame([{'dr_id': x['dr_id'], 'type': x['dr_type'], 'step': x['step'], 'actor': x['actor']['id'], 'model': x['actor'].get('model',''), 'summary': x['summary']}
                   for x in recs if not x['portfolio_scope'] or 'MTRS-2050' in x['portfolio_scope'] or x['step'].endswith('cma-judge')])
ev

,dr_id,type,step,actor,model,summary
0,dr_457dd5e494634d189425,decision,preflight.entity_regime,orchestrator,,"Entities ['sec_ria', 'registered_fund', 'erisa..."
1,dr_f695723f932a460c83c6,signal,data.snapshot,orchestrator,,Snapshot snap_3f592900a4dda39a (SYNTHETIC — pi...
2,dr_0c5ef00e163b4cbba4b6,signal,macro-regime,macro-regime,offline-deterministic,Regime: late_cycle (confidence 0.69)
3,dr_f0d06b415f5c4017bd2f,signal,covariance,orchestrator,,"Ledoit-Wolf covariance, shrinkage 0.01, 334 mo..."
4,dr_aad645c9e89c45ed89e0,decision,ac-us_large_cap.cma-judge,asset-class-analyst,offline-deterministic,US Large Cap Equity: E[r] 6.73% (auto-blend 7....
5,dr_1d4279e403fe4a87bd74,decision,ac-us_small_mid_cap.cma-judge,asset-class-analyst,offline-deterministic,US Small/Mid Cap Equity: E[r] 7.59% (auto-blen...
6,dr_efdb528cd21c4b66abac,decision,ac-intl_developed_eq.cma-judge,asset-class-analyst,offline-deterministic,International Developed Equity: E[r] 7.19% (au...
7,dr_2e6abbf21cdc48da9ecd,decision,ac-emerging_eq.cma-judge,asset-class-analyst,offline-deterministic,Emerging Markets Equity: E[r] 8.32% (auto-blen...
8,dr_63b43936326842d8baac,decision,ac-global_reits.cma-judge,asset-class-analyst,offline-deterministic,Global Real Estate (REITs): E[r] 6.72% (auto-b...
9,dr_bb22635f4f634459b765,decision,ac-commodities.cma-judge,asset-class-analyst,offline-deterministic,Diversified Commodities: E[r] 3.78% (auto-blen...
